# Camada Bronze — `financeiro`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Lançamentos financeiros e inadimplência (delimitadores mistos '|' e ';')  
**Arquivo bruto:** `Atividade11 - Financeiro aluno.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - Financeiro aluno.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - Financeiro aluno.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='financeiro',
    filename='Atividade11 - Financeiro aluno.txt',
    sep='|',
    mixed_sep=True,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'financeiro', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 1,000,000 | Tamanho bruto: 51.2091 MB
IDs únicos de aluno: 14,001 | Duplicatas exatas: 0
Salvo em: financeiro.parquet e financeiro_profile.json


,id_lancamento,id_aluno,competencia,valor_mensalidade,valor_desconto,valor_pago,situacao_financeira,forma_pagamento,data_pagamento,_bronze_raw_delimiter,_bronze_source_file,_bronze_ingested_at
0,1,6201,202402,1637.59,0.00,0.00,PENDENTE,,,|,Atividade11 - Financeiro aluno.txt,2026-11-13T18:55:24Z
1,2,14704,202310,2187.21,199.51,1987.70,PAGO,BOLETO,2024-01-26,;,Atividade11 - Financeiro aluno.txt,2026-11-13T18:55:24Z
2,3,12771,202307,1279.42,59.02,0.00,EM_ATRASO,,,|,Atividade11 - Financeiro aluno.txt,2026-11-13T18:55:24Z
3,4,11617,202310,1242.14,37.12,0.00,PENDENTE,,,|,Atividade11 - Financeiro aluno.txt,2026-11-13T18:55:24Z
4,5,14037,202406,1179.06,0.00,0.00,PENDENTE,,,|,Atividade11 - Financeiro aluno.txt,2026-11-13T18:55:24Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_lancamento": 0,
  "id_aluno": 1213,
  "competencia": 1253,
  "valor_mensalidade": 1128,
  "valor_desconto": 1272,
  "valor_pago": 1250,
  "situacao_financeira": 1284,
  "forma_pagamento": 748149,
  "data_pagamento": 748110
}

=== Anomalias detectadas para tratamento na Silver ===
{
  "situacao_distribuicao": {
    "PENDENTE": 252869,
    "PAGO": 251874,
    "EM_ATRASO": 247216,
    "CANCELADO": 246757,
    "": 1284
  },
  "delimitadores_detectados": {
    "|": 995100,
    ";": 4900
  }
}
